# FINRISK · Preparación de datos para la entrega 2

Nueva recuperación de fuentes públicas del 7 de noviembre de 2026. Este notebook revisa la base financiera reconstruida y los inventarios de insolvencia. La etiqueta a doce meses está pendiente de validación. Los resultados que siguen son descriptivos.


In [1]:
from pathlib import Path
import json
import os
import pandas as pd

candidatas = [Path.cwd(), *Path.cwd().parents]
RAIZ = next((p for p in candidatas if (p / "requirements.txt").is_file() and (p / "src").is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra el notebook desde FINRISK_COLOMBIA o su carpeta notebooks. Revise docs/entrega2/preparacion_datos.md.")
RAIZ_DATOS = Path(os.environ.get("FINRISK_DATOS_RAIZ", str(RAIZ))).expanduser().resolve()
REPORTES = RAIZ_DATOS / "reports/preparacion"
requeridos = [REPORTES / "adquisicion_financieros.json", REPORTES / "auditoria_insolvencia.json", REPORTES / "eventos_observados_insolvencia.csv", RAIZ_DATOS / "data/processed/recuperacion/base_financiera_recuperada_sin_etiquetas.csv.gz"]
faltantes = [p.relative_to(RAIZ_DATOS).as_posix() for p in requeridos if not p.is_file()]
if faltantes:
    raise FileNotFoundError("Faltan archivos de preparación. Ejecute descarga financiera, descarga de insolvencia, auditoría y consolidación. Siga docs/entrega2/preparacion_datos.md. Archivos faltantes: " + ", ".join(faltantes))
pd.set_option("display.max_columns", 12)


## Fuentes y versión

Los estados financieros provienen de los conjuntos [situación financiera](https://www.datos.gov.co/d/pfdp-zks5) y [resultado integral](https://www.datos.gov.co/d/prwj-nzxa). Se recuperan los conceptos contables usados por el proyecto y los periodos actuales de 2019 a 2024. Las columnas originales y los identificadores de fila se conservan en archivos comprimidos. Las URLs y huellas están en el manifiesto.

Los [inventarios de insolvencia](https://www.supersociedades.gov.co/web/procedimientos-de-insolvencia/informes) son cortes administrativos. Su unión ayuda a identificar antecedentes observados. La fecha actual de admisión requiere revisión para interpretarla como primera admisión histórica.


In [2]:
adquisicion = json.loads((REPORTES / "adquisicion_financieros.json").read_text(encoding="utf-8"))
assert adquisicion["estado"] == "completo", "La descarga debe terminar antes del análisis."
fuentes = []
for nombre, fuente in adquisicion["fuentes"].items():
    assert fuente["estado"] == "completo"
    archivos = fuente["archivos"]
    fuentes.append({"fuente": nombre, "fragmentos": len(archivos), "filas": sum(a["filas"] for a in archivos.values()), "MB_comprimidos": round(sum(a["bytes_gzip"] for a in archivos.values()) / 1_000_000, 2)})
pd.DataFrame(fuentes)


,fuente,fragmentos,filas,MB_comprimidos
0,resultados,27,1196115,37.04
1,situacion,33,1507002,48.22


In [3]:
base = pd.read_csv(RAIZ_DATOS / "data/processed/recuperacion/base_financiera_recuperada_sin_etiquetas.csv.gz", dtype={"nit": str})
assert not base.duplicated(["nit", "anio"]).any()
indicadores = ["liquidez_corriente", "endeudamiento", "roa", "margen_neto"]
cobertura = base.groupby("anio").agg(empresas=("nit", "nunique"), filas=("nit", "size"), cuatro_indicadores=("indicadores_completos", "sum"), con_conflictos=("conceptos_en_conflicto", lambda s: int(s.gt(0).sum())))
cobertura


,empresas,filas,cuatro_indicadores,con_conflictos
anio,,,,
2019,25610,25610,23622,38
2020,29949,29949,26586,79
2021,29200,29200,26705,83
2022,31149,31149,28442,271
2023,31396,31396,28791,134
2024,29362,29362,27279,108


## Calidad financiera

La consolidación aplica las prioridades de reporte y concepto del código original. Cuando importes de mejor prioridad discrepan, se conserva el conflicto en la auditoría y el importe queda ausente. Esta decisión cambia la versión respecto de la primera entrega. Las fechas de corte contable se conservan. La selección por variable puede combinar reportes individuales o consolidados, radicados, taxonomías o fechas distintas. La columna requiere_revision_coherencia identifica combinaciones que necesitan revisar sus candidatos antes del modelado. La fecha de disponibilidad pública de cada reporte no está en estas fuentes y queda pendiente.

Los indicadores son razones. Endeudamiento, ROA y margen neto se multiplican por cien si se presentan como porcentajes. Liquidez se expresa en veces. Conservar valores negativos evita excluir automáticamente empresas con dificultades.


In [4]:
pd.DataFrame({"faltantes": base[indicadores].isna().sum(), "porcentaje_faltante": (base[indicadores].isna().mean() * 100).round(2)})


,faltantes,porcentaje_faltante
liquidez_corriente,3525,2.00
endeudamiento,1530,0.87
roa,1543,0.87
margen_neto,13810,7.82


In [5]:
pd.Series({"empresa_anio": len(base), "empresas_distintas": base.nit.nunique(), "patrimonio_negativo": int(base.patrimonio.lt(0).sum()), "empresa_anio_con_conflictos": int(base.conceptos_en_conflicto.gt(0).sum()), "empresa_anio_requiere_revision_coherencia": int(base.requiere_revision_coherencia.sum()), "sin_fecha_disponibilidad": int(base.fecha_disponibilidad.isna().sum())}, name="observaciones")


empresa_anio                                 176666
empresas_distintas                            37231
patrimonio_negativo                            7723
empresa_anio_con_conflictos                     713
empresa_anio_requiere_revision_coherencia      2490
sin_fecha_disponibilidad                     176666
Name: observaciones, dtype: int64

## Revisión de insolvencia

Los conteos siguientes se calculan sobre los cortes descargados. Un NIT ausente del inventario actual puede aparecer en uno antiguo. Una fecha de admisión posterior a una presencia anterior indica que debemos revisar el proceso y su historia antes de identificar la primera admisión. Estos conteos no representan positivos o negativos para entrenar.


In [6]:
insolvencia = json.loads((REPORTES / "auditoria_insolvencia.json").read_text(encoding="utf-8"))
campos_auditoria = ["nits_distintos_union_cortes", "nits_union_ausentes_ultimo_corte", "nits_con_fecha_auto_observada", "nits_con_fecha_auto_posterior_a_un_corte_de_presencia_anterior"]
pd.Series({k: insolvencia[k] for k in campos_auditoria}, name="auditoria")


nits_distintos_union_cortes                                       8075
nits_union_ausentes_ultimo_corte                                  2378
nits_con_fecha_auto_observada                                     5697
nits_con_fecha_auto_posterior_a_un_corte_de_presencia_anterior     863
Name: auditoria, dtype: int64

In [7]:
eventos = pd.read_csv(REPORTES / "eventos_observados_insolvencia.csv", dtype={"NIT": str})
assert eventos.NIT.is_unique
cruce = base.nit.isin(eventos.NIT)
pd.Series({"empresas_financieras_con_antecedente_en_algun_corte": base.loc[cruce, "nit"].nunique(), "empresa_anio_con_alguna_coincidencia": int(cruce.sum()), "empresa_anio_sin_coincidencia_observada": int((~cruce).sum())}, name="cruce_descriptivo")


empresas_financieras_con_antecedente_en_algun_corte      3189
empresa_anio_con_alguna_coincidencia                    13502
empresa_anio_sin_coincidencia_observada                163164
Name: cruce_descriptivo, dtype: int64

## Condiciones para entrenar

La etiqueta propuesta requiere una primera admisión verificable posterior a la evaluación y dentro de los doce meses siguientes. Los negativos necesitan seguimiento completo y cobertura suficiente. También hay que verificar qué información financiera estaba disponible en la fecha de evaluación. Los casos anteriores, del mismo día y sin seguimiento suficiente se separan.

La ausencia en las listas consultadas se conserva como ausencia de registro observado. No permite concluir por sí sola que una empresa estuvo libre de insolvencia.


In [8]:
preparacion = {"base_financiera_recuperada": True, "fuentes_insolvencia_auditadas": True, "fechas_disponibilidad_validadas": False, "historial_primera_admision_validado": False, "cobertura_negativos_validada": False, "etiqueta_a_doce_meses_validada": False, "modelos_entrenados": False}
pd.Series(preparacion, name="estado")


base_financiera_recuperada              True
fuentes_insolvencia_auditadas           True
fechas_disponibilidad_validadas        False
historial_primera_admision_validado    False
cobertura_negativos_validada           False
etiqueta_a_doce_meses_validada         False
modelos_entrenados                     False
Name: estado, dtype: bool

## Siguiente fase propuesta

Una vez validada la etiqueta y las fechas, comparar una referencia de prevalencia, regresión logística y Random Forest con las mismas particiones temporales. Ajustar imputación y transformaciones con entrenamiento. Revisar la repetición de empresas entre periodos y separar la evaluación de empresas conocidas y nuevas cuando corresponda. Comparar precisión promedio (AP), sensibilidad, precisión de alertas y matriz de confusión. Elegir el umbral usando validación antes de abrir la prueba final.

Registrar cada evaluación real en el servidor de Jairo, experimento `Proyecto-MIAD`, con commit, versión de datos, protocolo, modelo y resumen. La prueba con Iris acredita integración técnica. Las métricas de FINRISK se calcularán con su propia etiqueta validada.

El tablero predictivo previsto permitirá consultar la estimación y la alerta por empresa. El modelo calculará el riesgo durante los doce meses posteriores a la evaluación. La validación pendiente de la etiqueta y las fechas antecede al entrenamiento y al uso de predicciones en el tablero.
